## Step 1; Install libraries

In [22]:
%pip install -q peft==0.13.0 transformers==4.44.0
%pip install datasets

## Step 2: Load model and tokenizer

In [23]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "bigscience/bloomz-560m"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

## Step 3: Load and preprocess the dataset

In [24]:
data = load_dataset("Abirate/english_quotes", split="train[:10%]")

data = data.map(lambda x: tokenizer(x["quote"]), batched=True)

train_sample = data.select(range(5))
display (train_sample)

Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 5
})

## Step 4 and 5: Configure LoRA and apply it to the model

In [25]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, lora_config)
print(peft_model.print_trainable_parameters())

trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.0176
None


## Step 6 and 7: Training arguments and Trainer

In [26]:
import transformers
from transformers import TrainingArguments, Trainer
import os

output_directory = os.path.join("../cache/working", "peft_lab_outputs")

training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,
    num_train_epochs=5,
    use_cpu=True
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

Step,Training Loss


TrainOutput(global_step=5, training_loss=2.8077375411987306, metrics={'train_runtime': 84.7713, 'train_samples_per_second': 0.295, 'train_steps_per_second': 0.059, 'total_flos': 2313450086400.0, 'train_loss': 2.8077375411987306, 'epoch': 5.0})

## Step 8: Save the fine-tuned LoRA model

In [27]:
import time

time_now = int(time.time())

peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)

print(f"model saved to: {peft_model_path}")

model saved to: ../cache/working/peft_lab_outputs/peft_model_1776965192


## Step 9: Load the saved model for inference

In [28]:
from peft import PeftModel

loaded_model = PeftModel.from_pretrained(
    model,
    peft_model_path,
    is_trainable=False
)

## Step 10: Generate text

In [30]:
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")

outputs = loaded_model.generate(
    input_ids=inputs["input_ids"],
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7
)

print(tokenizer.batch_decode(outputs, slip_special_tokens=True))

['Two things are infinite:  I am ready to be fed, and at the moment of need I am ready to kill.”” But when I have one thing, and it is too little, then it’s time to kill it, to wait for what isn’t ready to happen']
